# Healthiness and Processing Scores Analysis (Strict Schema, PySpark)

This notebook computes composite healthiness or processing scores for foods using PySpark, with strict schema mapping and the same data source as the outlier analysis. It visualizes and saves results to output/HealthinessScores.

In [10]:
# Imports and Spark session
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
import os

spark = SparkSession.builder.appName("HealthinessProcessingScores").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.1


In [11]:
# Configuration
input_path = "output/nutritional_profiles"
fallback_file = "output/nutritional_profiles/part-00000-d5d8abf9-0202-406e-af46-ff13446ce22f-c000.snappy.parquet"
output_dir = "output/HealthinessScores"
os.makedirs(output_dir, exist_ok=True)

# Strict schema mapping (edit as needed for your dataset)
schema_columns = {
    'id': 'fdc_id',
    'name': 'food_description',
    'category': 'food_type',
    'energy': 'energy',
    'protein': 'protein',
    'carb': 'carbs',
    'fat': 'total_fat',
    'fiber': 'fiber',
    'sugar': 'sugars',
    'saturated_fat': 'saturated_fat',
    'processing_level': 'processing_level'  # 1=unprocessed, 4=ultra-processed (NOVA)
}

nutrient_cols = ['energy', 'protein', 'carb', 'fat', 'fiber', 'sugar', 'saturated_fat']

In [ ]:
# Data loading with strict schema
from pyspark.sql.types import DoubleType

try:
    df = spark.read.parquet(input_path)
except Exception:
    df = spark.read.parquet(fallback_file)

for std_col, orig_col in schema_columns.items():
    if orig_col in df.columns:
        df = df.withColumnRenamed(orig_col, std_col)

selected_cols = ['id', 'name', 'category'] + nutrient_cols
# Only include processing_level if it exists in the DataFrame
if 'processing_level' in df.columns:
    selected_cols.append('processing_level')
df = df.select(*[c for c in selected_cols if c in df.columns])

for col in nutrient_cols:
    if col in df.columns:
        df = df.withColumn(col, df[col].cast(DoubleType()))
# Add processing_level if missing, default to 1 (unprocessed)
if 'processing_level' not in df.columns:
    from pyspark.sql.functions import lit
    df = df.withColumn('processing_level', lit(1.0))
else:
    df = df.withColumn('processing_level', df['processing_level'].cast(DoubleType()))

df.cache()
df.show(5)

+-------+--------------------+------------+------+-------+-----+-----+-----+-----+-------------+
|     id|                name|    category|energy|protein| carb|  fat|fiber|sugar|saturated_fat|
+-------+--------------------+------------+------+-------+-----+-----+-----+-----+-------------+
|1109427|SMOKEY TERIYAKI B...|branded_food| 321.0|  35.71| 25.0| 8.93|  0.0| 25.0|         3.57|
|1110221|APPLE, BLUEBERRY ...|branded_food|  67.0|    0.0|16.67|  0.0|  1.1|13.33|          0.0|
|1113353|SOLID MILK & WHIT...|branded_food| 526.0|   5.26|60.53|31.58|  2.6|55.26|        18.42|
|1115775|PEANUT BUTTER PRO...|branded_food| 479.0|  23.94|38.03|26.76|  4.2|26.76|         4.23|
|1121550|GARLIC & PEPPER S...|branded_food|2500.0|    0.0|  0.0|  0.0| NULL| NULL|          0.0|
+-------+--------------------+------------+------+-------+-----+-----+-----+-----+-------------+
only showing top 5 rows


26/01/11 17:09:56 WARN CacheManager: Asked to cache already cached data.


In [13]:
# Compute NOVA processing score (1=unprocessed, 4=ultra-processed)
df = df.withColumn('processing_score', F.col('processing_level'))
df.select('name', 'processing_score').show(5)

TypeError: 'JavaPackage' object is not callable

In [ ]:
# Compute nutritional density score: (fiber + protein) / (energy + sugar + fat)
def density_score_expr():
    return (F.col('fiber') + F.col('protein')) / (F.col('energy') + F.col('sugar') + F.col('fat') + F.lit(1e-6))

df = df.withColumn('density_score', density_score_expr())
df.select('name', 'density_score').show(5)

In [ ]:
# Composite healthiness score: higher density, lower processing = healthier
# Example formula: healthiness_score = density_score - 0.25 * (processing_score - 1)
df = df.withColumn(
    'healthiness_score',
    F.col('density_score') - 0.25 * (F.col('processing_score') - 1)
)
df.select('name', 'density_score', 'processing_score', 'healthiness_score').show(5)

In [ ]:
# Collect results to Pandas for visualization
pdf = df.select('name', 'category', 'density_score', 'processing_score', 'healthiness_score').toPandas()

# Plot distribution of healthiness scores
plt.figure(figsize=(8, 4))
plt.hist(pdf['healthiness_score'], bins=20, color='mediumseagreen', edgecolor='black')
plt.title('Distribution of Healthiness Scores')
plt.xlabel('Healthiness Score')
plt.ylabel('Count')
plt.show()

# Bar chart: Top 10 healthiest foods
pdf_sorted = pdf.sort_values('healthiness_score', ascending=False)
plt.figure(figsize=(10, 5))
plt.barh(pdf_sorted['name'].head(10)[::-1], pdf_sorted['healthiness_score'].head(10)[::-1], color='royalblue')
plt.xlabel('Healthiness Score')
plt.title('Top 10 Healthiest Foods')
plt.show()

# Bar chart: Top 10 most processed foods
pdf_proc = pdf.sort_values('processing_score', ascending=False)
plt.figure(figsize=(10, 5))
plt.barh(pdf_proc['name'].head(10)[::-1], pdf_proc['processing_score'].head(10)[::-1], color='tomato')
plt.xlabel('Processing Score (NOVA)')
plt.title('Top 10 Most Processed Foods')
plt.show()

In [ ]:
# Save results as CSV and Parquet
pdf_sorted.to_csv(f"{output_dir}/healthiness_scores.csv", index=False)
df.write.mode('overwrite').parquet(f"{output_dir}/healthiness_scores.parquet")
print(f"Results saved to {output_dir}")